# Build prediction datasets (CPU, cluster)

Knits batchwise GCP embeddings into analysis-ready files, then builds the embedding-prediction
datasets **once per anchor** (treatment and sequencing — see `anchors.py`), then builds the
SLURM manifests per anchor. Ends by printing the `sbatch` lines to launch full-cohort and
feature-comparison training next.

In [ ]:
from __future__ import annotations

import subprocess
import sys
from pathlib import Path


def find_v2_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "config.py").is_file() and (candidate / "pipelines").is_dir():
            return candidate
    raise RuntimeError(f"Could not find v2 root from {start}")


V2_ROOT = find_v2_root()
if str(V2_ROOT) not in sys.path:
    sys.path.insert(0, str(V2_ROOT))

from anchors import ANCHORS

ANCHOR_LIST = sorted(ANCHORS.keys())

print(f"v2 root: {V2_ROOT}")
print(f"Python:  {sys.executable}")
print(f"Anchors: {ANCHOR_LIST}")

## Knit embeddings

Anchor-independent: reads raw per-batch GCP embeddings and metadata, writes the full
analysis-ready embedding array + metadata used by every downstream anchor.

In [ ]:
def run(args: list[str]) -> None:
    print("\n=== " + " ".join(args) + " ===", flush=True)
    subprocess.run([sys.executable, "-m", *args], cwd=V2_ROOT, check=True)


run(["pipelines.preprocessing.knit_embeddings"])
run(["pipelines.preprocessing.report_data_availability"])

## Build embedding-prediction datasets, once per anchor

In [ ]:
for anchor in ANCHOR_LIST:
    run(["pipelines.preprocessing.generate_embedding_prediction_datasets", "--anchor", anchor])

## Build SLURM manifests, once per anchor

In [ ]:
for anchor in ANCHOR_LIST:
    run(["pipelines.training.build_slurm_manifests", "--anchor", anchor])

print("\nDone.")

## Next: launch training

Manifests are anchor-suffixed (`anchor_suffix()` in `anchors.py`; empty for the default
`treatment` anchor). Run these from the cluster shell, one anchor at a time:

In [ ]:
for anchor in ANCHOR_LIST:
    print(f"# anchor={anchor}")
    print(f"ANCHOR={anchor} bash v2/slurm/launch_full_cohort.sh")
    print(f"ANCHOR={anchor} bash v2/slurm/launch_feature_comp.sh")
    print(f"# (after full_cohort completes) ANCHOR={anchor} bash v2/slurm/launch_full_cohort_risk_scores.sh")
    print()